In [1]:
import pandas as pd
from collections import defaultdict

# --- 1. Load -------------------------------------------------------------
def load(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()           # guard against header whitespace
    key_cols = ["clientProject", "clientProjectOrganisation",
                "dependencyGroupID", "dependencyArtifactID",
                "previousVersion", "newVersion"]
    for c in key_cols:
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip()
    return df


# The identity of a project + the dependency it consumes.
# Same (project, org, group, artifact) across DIFFERENT version pairs = a timeline.
GROUP = ["clientProject", "clientProjectOrganisation",
         "dependencyGroupID", "dependencyArtifactID"]


# --- 2. Assign a stable group id, keep only multi-update groups ----------
def tag_groups(df):
    df = df.copy()
    df["group_id"] = df.groupby(GROUP, dropna=False).ngroup()

    # count DISTINCT version transitions per group (dedupe exact repeats first)
    n_updates = (df.drop_duplicates(GROUP + ["previousVersion", "newVersion"])
                   .groupby(GROUP).size().rename("n_updates"))
    df = df.merge(n_updates, on=GROUP, how="left")
    return df


def longitudinal_only(df, min_updates=2):
    """Groups that saw >= min_updates distinct version bumps."""
    return df[df["n_updates"] >= min_updates].copy()


# --- 3. Order each group into an upgrade chain ---------------------------
# Each row is an edge previousVersion -> newVersion. A "start" is a
# previousVersion that never appears as anyone's newVersion. Walk forward
# from each start; anything left over (cycles / forks) gets step = -1.
def order_chain(g):
    rows = g.to_dict("records")
    new_versions = {r["newVersion"] for r in rows}
    by_prev = defaultdict(list)
    for r in rows:
        by_prev[r["previousVersion"]].append(r)

    starts = [r for r in rows if r["previousVersion"] not in new_versions]
    if not starts:                      # pure cycle: fall back to timestamp
        starts = sorted(rows, key=lambda r: r.get("execution_timestamp", ""))[:1]

    ordered, seen = [], set()

    def walk(r, step):
        edge = (r["previousVersion"], r["newVersion"])
        if edge in seen:
            return
        seen.add(edge)
        r = {**r, "step": step}
        ordered.append(r)
        for nxt in by_prev.get(r["newVersion"], []):
            walk(nxt, step + 1)

    for s in starts:
        walk(s, 0)
    for r in rows:                      # leftovers (forks / disconnected)
        edge = (r["previousVersion"], r["newVersion"])
        if edge not in seen:
            seen.add(edge)
            ordered.append({**r, "step": -1})

    return pd.DataFrame(ordered)


def add_chain_step(df):
    """Append a chain_step column WITHOUT dropping rows or reordering.
    All original columns/rows stay intact; step is looked up per edge."""
    df = df.copy()
    step_lookup = {}
    for keys, g in df.groupby(GROUP, dropna=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        ordered = order_chain(g)
        for _, r in ordered.iterrows():
            step_lookup[keys + (r["previousVersion"], r["newVersion"])] = r["step"]

    df["chain_step"] = df.apply(
        lambda row: step_lookup.get(
            tuple(row[c] for c in GROUP) + (row["previousVersion"], row["newVersion"])),
        axis=1)
    return df


# --- 4. Per-group summary for the longitudinal view ----------------------
def summarize(long_df):
    def agg(g):
        return pd.Series({
            "n_updates": g[["previousVersion", "newVersion"]].drop_duplicates().shape[0],
            "version_span": f'{g["previousVersion"].iloc[0]} .. {g["newVersion"].iloc[-1]}',
            "n_breaking": int((g.get("docker_image_breaking", pd.Series(dtype=bool))
                               .astype(str).str.lower().isin(["true", "1"])).sum())
                          if "docker_image_breaking" in g else None,
            "failure_categories": ", ".join(sorted(
                g["failureCategory"].dropna().astype(str).unique()))
                if "failureCategory" in g else "",
            "success_rate": round(g.get("execution_success", pd.Series(dtype=float))
                                   .astype(str).str.lower().isin(["true", "1"]).mean(), 3)
                            if "execution_success" in g else None,
        })
    return (long_df.groupby(GROUP, dropna=False).apply(agg)
                   .reset_index()
                   .sort_values("n_updates", ascending=False))


def longitudinal_customid_groups(df):
    """One row per longitudinal cohort, listing its custom_ids in upgrade
    order (by chain_step). These are the custom_ids that 'go together'."""
    long_df = df[df["is_longitudinal"]].sort_values(GROUP + ["chain_step"])
    rows = []
    for (proj, org, gid, aid), g in long_df.groupby(GROUP, dropna=False):
        rows.append({
            "group_id":   int(g["group_id"].iloc[0]),
            "clientProject": proj,
            "org":        org,
            "dependency": f"{gid}:{aid}",
            "n_updates":  int(g["n_updates"].iloc[0]),
            "custom_ids": " -> ".join(g["custom_id"].astype(str)),
        })
    out = (pd.DataFrame(rows)
             .sort_values("n_updates", ascending=False)
             .reset_index(drop=True))
    out["group_id"] = out.index + 1          # 1 = longest progression
    return out


# --- 5. Run --------------------------------------------------------------
if __name__ == "__main__":
    df = load("/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes_with_categories_v2.csv")

    # Enrich the ORIGINAL frame: keep every row and every column intact,
    # just append the analysis columns.
    df = tag_groups(df)                 # + group_id, n_updates
    df = add_chain_step(df)             # + chain_step (timeline position)
    df["is_longitudinal"] = df["n_updates"] >= 2

    df.to_csv("enriched_output.csv", index=False)   # <-- all originals + new cols

    # console sanity check
    print(f"rows in  ............. {len(df)}")
    print(f"columns out .......... {len(df.columns)} "
          f"(added: group_id, n_updates, chain_step, is_longitudinal)")
    print(f"groups (project+dep) . {df['group_id'].nunique()}")
    print(f"longitudinal groups .. {df.loc[df['is_longitudinal'], 'group_id'].nunique()}")
    print(f"rows in those groups . {int(df['is_longitudinal'].sum())}")

    # the custom_ids that are longitudinal together, one cohort per row
    cid_groups = longitudinal_customid_groups(df)
    cid_groups.to_csv("longitudinal_customid_groups.csv", index=False)
    print("\n--- longitudinal cohorts (custom_ids in upgrade order) ---")
    print(cid_groups.to_string(index=False))

rows in  ............. 89
columns out .......... 34 (added: group_id, n_updates, chain_step, is_longitudinal)
groups (project+dep) . 38
longitudinal groups .. 14
rows in those groups . 64

--- longitudinal cohorts (custom_ids in upgrade order) ---
 group_id          clientProject                                   org                                  dependency  n_updates                                                               custom_ids
        1 IDS-Messaging-Services International-Data-Spaces-Association               org.springframework:spring-tx          8 BBC05 -> BBC104 -> BBC107 -> BBC170 -> BBC186 -> BBC35 -> BBC53 -> BBC81
        2   jasmine-maven-plugin                                searls                         org.slf4j:slf4j-api          7           BBC135 -> BBC178 -> BBC187 -> BBC26 -> BBC66 -> BBC88 -> BBC98
        3                recheck                                retest                         org.slf4j:slf4j-api          7         BBC103 -> BBC131 -> B

In [4]:
import pandas as pd

# ---------------------------------------------------------------------------
# Inputs
#   ENRICHED_CSV : output of script #1 (group_id, chain_step, is_longitudinal)
#   ERROR_CSV    : error-details CSV. Ground truth = bump_bc_errors,
#                  LLM detection = llm_detected_errors.
# Join key: custom_id (stripped on BOTH sides — that was the earlier bug).
#
# bump_bc_errors is identical for a custom_id across all model/variant rows,
# so it is taken from ANY row. llm_detected_errors is restricted to the
# context_variant below (and optional MODELS). If a custom_id has no such
# LLM row, the model failed to capture the BC -> shown as "not captured".
# ---------------------------------------------------------------------------
ENRICHED_CSV    = "enriched_output.csv"
ERROR_CSV       = "detected_bc_errortype_coverage.csv"   
REPORT_TXT      = "longitudinal_error_progression.txt"
TIDY_CSV        = "longitudinal_error_progression_tidy.csv"

CONTEXT_VARIANT = "class"                     # only this variant is analysed
MODELS          = None                        # e.g. ["gpt-4o"]; None = all models
GT_COL          = "bump_bc_errors"
LLM_COL         = "llm_detected_errors"

GROUP = ["clientProject", "clientProjectOrganisation",
         "dependencyGroupID", "dependencyArtifactID"]


def _s(v, dash="—"):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return dash
    t = str(v).strip()
    return dash if t in ("", "nan", "None") else t


def _t(s, n=90):
    s = str(s)
    return s if len(s) <= n else s[:n - 1] + "…"


def load(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    if "custom_id" in df.columns:                       # <- the fix
        df["custom_id"] = df["custom_id"].astype(str).str.strip()
    return df


def constancy(seq):
    uniq = list(dict.fromkeys(seq))
    real = [u for u in uniq if u != "—"]
    return ("CONSTANT" if len(set(real)) <= 1 else "CHANGES"), uniq


def build_report(df, err):
    long_df = df[df["is_longitudinal"]].copy()
    long_df["custom_id"] = long_df["custom_id"].astype(str).str.strip()
    long_df["chain_step"] = pd.to_numeric(long_df["chain_step"], errors="coerce")
    long_df = long_df.sort_values(GROUP + ["chain_step"])

    # ground truth: one bump_bc_errors per custom_id, from ANY row
    gt_by_cid = {}
    for cid, g in err.groupby("custom_id"):
        real = [_s(v) for v in g[GT_COL] if _s(v) != "—"]
        gt_by_cid[cid] = real[0] if real else "—"

    # LLM detections: restricted to the chosen context_variant (+ models)
    llm = err[err["context_variant"].astype(str).str.strip().str.lower()
              == CONTEXT_VARIANT.lower()]
    if MODELS:
        llm = llm[llm["model"].isin(MODELS)]
    llm_by_cid = {cid: g for cid, g in llm.groupby("custom_id")}

    # diagnostics
    cids = set(long_df["custom_id"])
    print(f"longitudinal custom_ids ........ {len(cids)}")
    print(f"  with ground-truth (bump) ..... {len(cids & set(gt_by_cid))}")
    print(f"  with class-variant LLM row ... {len(cids & set(llm_by_cid))}")

    order = (long_df.groupby(GROUP)["n_updates"].first()
                    .sort_values(ascending=False).index)

    lines, tidy = [], []
    for keys in order:
        sub = long_df
        for col, val in zip(GROUP, keys):
            sub = sub[sub[col] == val]
        sub = sub.sort_values("chain_step")
        proj, org, gid, aid = keys
        group_id = int(sub["group_id"].iloc[0])
        steps = sub.to_dict("records")

        norm_seq = [_s(r.get("normalized_error_category")) for r in steps]
        gt_seq   = [gt_by_cid.get(r["custom_id"], "—") for r in steps]
        n_lab, n_uniq = constancy(norm_seq)
        g_lab, g_uniq = constancy(gt_seq)
        flag = " " if "CHANGES" in (n_lab, g_lab) else " "

        lines += [
            "=" * 90,
            f"GROUP {group_id} · {len(steps)} steps · {_s(proj)} ({_s(org)})",
            f"dependency : {_s(gid)}:{_s(aid)}",
            f"{flag}normalized error category   : {n_lab}   [{' → '.join(_t(u,60) for u in n_uniq)}]",
            f"{flag}ground-truth bump_bc_errors : {g_lab}   [{' → '.join(_t(u,60) for u in g_uniq)}]",
            "-" * 90,
        ]

        for r in steps:
            cid = r["custom_id"]
            step = r["chain_step"]
            step_s = str(int(step)) if pd.notna(step) else "?"
            ver = f'{_s(r.get("previousVersion"))} → {_s(r.get("newVersion"))}'
            gt = gt_by_cid.get(cid, "—")

            lines.append(f"[step {step_s}] {cid}   {ver}")
            lines.append(f"   ground truth (bump) : {gt}")
            lines.append(f"   normalized          : {_s(r.get('normalized_error_category'))}")

            g = llm_by_cid.get(cid)
            if g is None or g.empty:
                lines.append(f"   LLM [{CONTEXT_VARIANT}]         : not captured (no {CONTEXT_VARIANT}-variant row)")
                tidy.append({
                    "group_id": group_id, "chain_step": step_s, "custom_id": cid,
                    "dependency": f"{_s(gid)}:{_s(aid)}",
                    "previousVersion": _s(r.get("previousVersion")),
                    "newVersion": _s(r.get("newVersion")),
                    "bump_bc_errors": gt, "model": "—",
                    "context_variant": CONTEXT_VARIANT,
                    "llm_detected_errors": "NOT_CAPTURED", "match_rate_%": "—",
                })
            else:
                for _, m in g.iterrows():
                    model = _s(m.get("model"))
                    detected = _s(m.get(LLM_COL))
                    lines.append(f"   LLM {model} [{CONTEXT_VARIANT}] : {detected}")
                    tidy.append({
                        "group_id": group_id, "chain_step": step_s, "custom_id": cid,
                        "dependency": f"{_s(gid)}:{_s(aid)}",
                        "previousVersion": _s(r.get("previousVersion")),
                        "newVersion": _s(r.get("newVersion")),
                        "bump_bc_errors": gt, "model": model,
                        "context_variant": CONTEXT_VARIANT,
                        "llm_detected_errors": detected,
                    })
            lines.append("")
        lines.append("")

    return "\n".join(lines), pd.DataFrame(tidy)


# --- Run -------------------------------------------------------------------
if __name__ == "__main__":
    df  = load(ENRICHED_CSV)
    err = load(ERROR_CSV)

    report, tidy = build_report(df, err)

    with open(REPORT_TXT, "w", encoding="utf-8") as f:
        f.write(report)
    tidy.to_csv(TIDY_CSV, index=False)

    print(report)
    print(f"\n[written] {REPORT_TXT}  ·  {TIDY_CSV} ({len(tidy)} rows)")

longitudinal custom_ids ........ 64
  with ground-truth (bump) ..... 23
  with class-variant LLM row ... 22
GROUP 2 · 8 steps · IDS-Messaging-Services (International-Data-Spaces-Association)
dependency : org.springframework:spring-tx
 normalized error category   : CONSTANT   [BeanInstantiationException|MojoFailureException|SocketTimeo…]
 ground-truth bump_bc_errors : CONSTANT   [—]
------------------------------------------------------------------------------------------
[step 0] BBC05   5.3.24 → 6.0.3
   ground truth (bump) : —
   normalized          : BeanInstantiationException|MojoFailureException|SocketTimeoutException|UnsupportedClassVersionError
   LLM [class]         : not captured (no class-variant row)

[step 0] BBC104   5.3.24 → 6.0.2
   ground truth (bump) : —
   normalized          : BeanInstantiationException|MojoFailureException|SocketTimeoutException|UnsupportedClassVersionError
   LLM [class]         : not captured (no class-variant row)

[step 0] BBC107   5.3.24 → 6.0.